# 01 — Merge and harmonize the four post-QC datasets

This notebook creates the full-gene merged object used by the downstream
main-analysis notebooks.

Datasets:

- DS1 / GSE103224
- DS2 / GSE141383
- DS3 / GSE182109
- DS4 / GSE278450

Scientific behavior retained from the original workflow:

1. DS1 and DS2 Ensembl identifiers are mapped with GENCODE v49.
2. DS3 and DS4 use gene symbols already stored in `var_names`.
3. Features mapping to the same gene symbol are summed.
4. Only gene symbols shared by all four datasets are retained.
5. Raw counts are stored in both `X` and `layers["counts"]`.
6. All post-QC cells are retained.

Memory-safety changes:

- matrices are read directly from HDF5 CSR storage;
- harmonization and common-gene subsetting are performed in cell-row chunks;
- each chunk is written to disk immediately;
- `anndata.experimental.concat_on_disk` is used without an in-memory fallback;
- final validation reads metadata directly from HDF5 and never loads the
  complete expression matrices into RAM;
- `concat_on_disk` runs in a short-lived child process;
- all chunk files and assembled temporary H5ADs are written to the internal
  NVMe scratch drive (default `E:`), while durable outputs remain under the
  repository on `G:`;
- completed H5ADs are copied to `G:` only once, sequentially;
- `concat_on_disk` is allowed to omit `uns` on this AnnData version;
- provenance is patched into the completed scratch H5AD by a second,
  short-lived child process that opens the file in `r+` and then exits,
  avoiding persistent Windows HDF5 locks.

Main output:

`data/processed/integrated/GBM_core_4datasets_fullgene_postQC_merged_counts.h5ad`

## Reproducibility correction in v6

The archived executed Ubuntu notebook confirms the historical outputs:

- DS1 harmonized genes: 23,982
- DS2 harmonized genes: 21,840
- DS3 harmonized genes: 30,284
- DS4 harmonized genes: 24,584
- four-way common genes: 15,176
- final merged shape: 351,427 × 15,176

The Windows reconstruction initially produced 25,769 harmonized genes for DS2 because the rebuilt DS2 object stores gene symbols in `var_names` while preserving the original Ensembl identifiers in `var['ensembl_id_version']`. v6 explicitly resolves the Ensembl identifier source before GENCODE v49 mapping, matching the historical analysis semantics.

## v7 compatibility fix

v6 contained an accidental double escape in the Ensembl-ID validation regular expression (`\\d` was interpreted literally). v7 corrects the detector to `^ENSG\d+$`. No scientific rule or mapping source is changed; the fix only allows genuine Ensembl identifiers to be recognized.


In [1]:
# =========================
# 0) Environment
# =========================

import gc
import gzip
import hashlib
import inspect
import json
import os
import re
import shutil
import subprocess
import sys
import time
from copy import deepcopy
from itertools import combinations
from pathlib import Path

import anndata as ad
import h5py
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse

try:
    from anndata.io import read_elem
except ImportError:
    # Compatibility with older AnnData releases.
    from anndata._io.specs import read_elem

print("Python:", sys.executable)
print("scanpy:", sc.__version__)
print("anndata:", ad.__version__)
print("pandas:", pd.__version__)


Python: C:\Users\AmirSilco\.conda\envs\glioma-cnv-atlas\python.exe
scanpy: 1.11.5
anndata: 0.12.7
pandas: 2.3.3


C:\Users\AmirSilco\AppData\Local\Temp\ipykernel_36540\2745777760.py:34: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  print("scanpy:", sc.__version__)
C:\Users\AmirSilco\AppData\Local\Temp\ipykernel_36540\2745777760.py:35: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  print("anndata:", ad.__version__)


In [2]:
# =========================
# 1) Project paths and configuration
# =========================

def find_repository_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    start = start.expanduser().resolve()

    for candidate in (start, *start.parents):
        if (
            (candidate / "README.md").exists()
            and (candidate / "code").exists()
        ):
            return candidate

    return start


PROJECT_DIR = Path(
    os.environ.get(
        "GLIOMA_PROJECT_ROOT",
        find_repository_root(),
    )
).expanduser().resolve()

REFERENCE_DIR = PROJECT_DIR / "data" / "reference"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
METADATA_DIR = PROJECT_DIR / "metadata"

INTEGRATED_DIR = PROCESSED_DIR / "integrated"
HARM_DIR = (
    PROCESSED_DIR
    / "core_4datasets_harmonized_gene_symbols"
)
COMMON_DIR = HARM_DIR / "common_genes"

# Fast transient storage. The repository can stay on the large USB G: drive,
# while expensive chunk creation and concat_on_disk assembly happen on the
# internal NVMe. Override with GLIOMA_SCRATCH_ROOT if needed.
_default_scratch = Path(r"E:\glioma-cnv-scratch\main_analysis_01")
SCRATCH_ROOT = Path(
    os.environ.get(
        "GLIOMA_SCRATCH_ROOT",
        str(_default_scratch),
    )
).expanduser().resolve()

WORK_DIR = SCRATCH_ROOT / "work"
STAGE_DIR = SCRATCH_ROOT / "stage"

for directory in (
    PROCESSED_DIR,
    METADATA_DIR,
    INTEGRATED_DIR,
    HARM_DIR,
    COMMON_DIR,
    SCRATCH_ROOT,
    WORK_DIR,
    STAGE_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)

scratch_usage = shutil.disk_usage(SCRATCH_ROOT)
SCRATCH_FREE_GIB = scratch_usage.free / (1024 ** 3)

if SCRATCH_FREE_GIB < 20:
    raise OSError(
        "The NVMe scratch drive needs at least 20 GiB free for this notebook. "
        f"Observed {SCRATCH_FREE_GIB:.1f} GiB at {SCRATCH_ROOT}."
    )


GTF_PATH = (
    REFERENCE_DIR
    / "gencode.v49.primary_assembly.annotation.gtf.gz"
)

CORE_DATASETS = [
    {
        "core_dataset": "DS1_GSE103224",
        "dataset_id": "DS1",
        "geo_id": "GSE103224",
        "path": (
            PROCESSED_DIR
            / "DS1_GSE103224"
            / "GSE103224_postQC.h5ad"
        ),
        "gene_id_mode": "ensembl",
    },
    {
        "core_dataset": "DS2_GSE141383",
        "dataset_id": "DS2",
        "geo_id": "GSE141383",
        "path": (
            PROCESSED_DIR
            / "DS2_GSE141383"
            / "GSE141383_postQC.h5ad"
        ),
        "gene_id_mode": "ensembl",
    },
    {
        "core_dataset": "DS3_GSE182109",
        "dataset_id": "DS3",
        "geo_id": "GSE182109",
        "path": (
            PROCESSED_DIR
            / "DS3_GSE182109"
            / "GSE182109_postQC.h5ad"
        ),
        "gene_id_mode": "symbol",
    },
    {
        "core_dataset": "DS4_GSE278450",
        "dataset_id": "DS4",
        "geo_id": "GSE278450",
        "path": (
            PROCESSED_DIR
            / "DS4_GSE278450"
            / "GSE278450_postQC.h5ad"
        ),
        "gene_id_mode": "symbol",
    },
]

MIN_GENES_PER_INPUT = 10_000
MIN_COMMON_GENES_AFTER_HARMONIZATION = 10_000
EXPECTED_FINAL_SHAPE = (351_427, 15_176)

HISTORICAL_HARMONIZED_GENE_COUNTS = {
    "DS1_GSE103224": 23_982,
    "DS2_GSE141383": 21_840,
    "DS3_GSE182109": 30_284,
    "DS4_GSE278450": 24_584,
}

HISTORICAL_COMMON_GENE_COUNT = 15_176

# v6 can safely reuse a v5 cache only if the source fingerprint, expected
# shape, and exact var-name hash also match. This lets DS1/DS3/DS4 reuse
# valid harmonized outputs, while DS2 is automatically rebuilt because the
# corrected GENCODE mapping changes its expected gene set.
COMPATIBLE_CACHE_VERSIONS = {
    "main-analysis-01-nvme-scratch-v5-child-uns-patch",
    "main-analysis-01-nvme-scratch-v7-ensembl-regex-fix",
}

# The four input objects are already the selected core datasets.
FILTER_TO_CORE_CANDIDATES_IF_PRESENT = False

# Reduce this if a row chunk still creates memory pressure.
ROW_CHUNK_SIZE = int(
    os.environ.get(
        "GLIOMA_MERGE_ROW_CHUNK_SIZE",
        "1024",
    )
)

# Limits the amount loaded by concat_on_disk when the installed
# AnnData version exposes max_loaded_elems.
CONCAT_MAX_LOADED_ELEMS = int(
    os.environ.get(
        "GLIOMA_CONCAT_MAX_LOADED_ELEMS",
        "3000000",
    )
)

CACHE_VERSION = (
    "main-analysis-01-nvme-scratch-v6-ds2-ensembl-source-fix"
)

FINAL_MERGED_H5AD = (
    INTEGRATED_DIR
    / "GBM_core_4datasets_fullgene_postQC_merged_counts.h5ad"
)

INPUT_INSPECTION_TSV = (
    METADATA_DIR
    / "core_4datasets_postQC_input_inspection_before_merge_corrected.tsv"
)
GENE_HARMONIZATION_TSV = (
    METADATA_DIR
    / "gene_harmonization_report_4datasets_corrected.tsv"
)
PAIRWISE_INTERSECTION_TSV = (
    METADATA_DIR
    / "gene_symbol_pairwise_intersections_4datasets_corrected.tsv"
)
COMMON_GENES_TSV = (
    METADATA_DIR
    / "common_genes_after_harmonization_4datasets_corrected.tsv"
)
COMMON_SUBSET_TSV = (
    METADATA_DIR
    / "common_gene_subset_report_4datasets_corrected.tsv"
)
MERGE_SUMMARY_TSV = (
    METADATA_DIR
    / "merged_fullgene_QC_summary_4datasets_corrected.tsv"
)
CELL_COUNTS_BY_DATASET_TSV = (
    METADATA_DIR
    / "merged_fullgene_cell_counts_by_dataset_4datasets_corrected.tsv"
)
CELL_COUNTS_BY_SAMPLE_TSV = (
    METADATA_DIR
    / "merged_fullgene_cell_counts_by_sample_4datasets_corrected.tsv"
)
CELL_COUNTS_BY_PATIENT_TSV = (
    METADATA_DIR
    / "merged_fullgene_cell_counts_by_patient_4datasets_corrected.tsv"
)

print("PROJECT_DIR:", PROJECT_DIR)
print("GTF_PATH:", GTF_PATH, "| exists:", GTF_PATH.exists())
print("ROW_CHUNK_SIZE:", ROW_CHUNK_SIZE)
print("SCRATCH_ROOT:", SCRATCH_ROOT)
print("SCRATCH_FREE_GiB:", round(SCRATCH_FREE_GIB, 1))
print("FINAL_MERGED_H5AD:", FINAL_MERGED_H5AD)

print("\nInput files:")
for item in CORE_DATASETS:
    print(
        "-",
        item["core_dataset"],
        item["path"],
        "| exists:",
        Path(item["path"]).exists(),
    )


PROJECT_DIR: G:\Repository\glioma-cnv-single-cell-atlas
GTF_PATH: G:\Repository\glioma-cnv-single-cell-atlas\data\reference\gencode.v49.primary_assembly.annotation.gtf.gz | exists: True
ROW_CHUNK_SIZE: 1024
SCRATCH_ROOT: E:\glioma-cnv-scratch\main_analysis_01
SCRATCH_FREE_GiB: 42.5
FINAL_MERGED_H5AD: G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_fullgene_postQC_merged_counts.h5ad

Input files:
- DS1_GSE103224 G:\Repository\glioma-cnv-single-cell-atlas\data\processed\DS1_GSE103224\GSE103224_postQC.h5ad | exists: True
- DS2_GSE141383 G:\Repository\glioma-cnv-single-cell-atlas\data\processed\DS2_GSE141383\GSE141383_postQC.h5ad | exists: True
- DS3_GSE182109 G:\Repository\glioma-cnv-single-cell-atlas\data\processed\DS3_GSE182109\GSE182109_postQC.h5ad | exists: True
- DS4_GSE278450 G:\Repository\glioma-cnv-single-cell-atlas\data\processed\DS4_GSE278450\GSE278450_postQC.h5ad | exists: True


In [3]:
# =========================
# 2) General helpers
# =========================

def remove_path(
    path: Path,
    attempts: int = 12,
    initial_delay_seconds: float = 0.25,
):
    path = Path(path)
    delay = float(initial_delay_seconds)
    last_error = None

    for attempt in range(1, attempts + 1):
        try:
            if path.is_dir():
                shutil.rmtree(path)
            elif path.exists():
                path.unlink()

            return

        except PermissionError as error:
            last_error = error
            gc.collect()

            if attempt == attempts:
                break

            print(
                "[Windows path-lock retry]",
                f"attempt {attempt}/{attempts}",
                path.name,
                f"sleep={delay:.2f}s",
            )
            time.sleep(delay)
            delay = min(delay * 1.5, 2.0)

    raise PermissionError(
        "Windows did not release the path lock after "
        f"{attempts} attempts: {path}"
    ) from last_error


def write_tsv_atomic(
    frame: pd.DataFrame,
    path: Path,
):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    temporary = path.with_name(
        f"{path.name}.temporary"
    )
    remove_path(temporary)

    frame.to_csv(
        temporary,
        sep="\t",
        index=False,
    )
    os.replace(temporary, path)

    print("[SAVED]", path, "shape=", frame.shape)


def write_json_atomic(
    payload: dict,
    path: Path,
):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    temporary = path.with_name(
        f"{path.name}.temporary"
    )
    remove_path(temporary)

    with temporary.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
        )

    os.replace(temporary, path)


def publish_file_cross_drive(
    scratch_file: Path,
    durable_file: Path,
):
    """
    Copy one completed H5AD from fast scratch to the durable repository.

    The copy is sequential. A '.copying' sibling is used on the destination
    drive so an interrupted transfer never appears as a valid final output.
    """
    scratch_file = Path(scratch_file)
    durable_file = Path(durable_file)
    durable_file.parent.mkdir(parents=True, exist_ok=True)

    if not scratch_file.exists():
        raise FileNotFoundError(scratch_file)

    destination_temporary = durable_file.with_name(
        f"{durable_file.name}.copying"
    )
    remove_path(destination_temporary)

    print(
        "[PUBLISH sequential copy]",
        scratch_file,
        "->",
        durable_file,
    )

    shutil.copyfile(
        scratch_file,
        destination_temporary,
    )

    if destination_temporary.stat().st_size != scratch_file.stat().st_size:
        remove_path(destination_temporary)
        raise AssertionError(
            "Cross-drive publication size mismatch."
        )

    remove_path(durable_file)
    os.replace(
        destination_temporary,
        durable_file,
    )


def clean_string_series(
    values,
    fill_value: str,
) -> pd.Series:
    series = pd.Series(values).astype(str).str.strip()

    bad = (
        series.eq("")
        | series.str.lower().isin(
            [
                "nan",
                "none",
                "null",
                "na",
                "n/a",
                "<na>",
            ]
        )
    )
    series.loc[bad] = fill_value

    return series


def strip_ensembl_version(values) -> pd.Series:
    return (
        pd.Series(values)
        .astype(str)
        .str.replace(
            r"\.\d+$",
            "",
            regex=True,
        )
    )


def is_ensembl_like(
    values,
    min_fraction: float = 0.5,
) -> bool:
    series = pd.Series(values).astype(str)

    if len(series) == 0:
        return False

    return (
        series
        .str.replace(
            r"\.\d+$",
            "",
            regex=True,
        )
        .str.startswith("ENSG")
        .mean()
        >= min_fraction
    )


def make_unique_index(
    values,
    separator: str = "-",
) -> pd.Index:
    used = set()
    next_suffix = {}
    output = []

    for raw_value in values:
        value = str(raw_value)

        if value not in used:
            used.add(value)
            next_suffix[value] = 1
            output.append(value)
            continue

        suffix = next_suffix.get(value, 1)
        candidate = f"{value}{separator}{suffix}"

        while candidate in used:
            suffix += 1
            candidate = f"{value}{separator}{suffix}"

        next_suffix[value] = suffix + 1
        used.add(candidate)
        output.append(candidate)

    return pd.Index(
        output,
        dtype="object",
    )


def strings_sha256(values) -> str:
    digest = hashlib.sha256()

    for value in values:
        digest.update(str(value).encode("utf-8"))
        digest.update(b"\0")

    return digest.hexdigest()


def file_fingerprint(path: Path) -> dict:
    path = Path(path).resolve()
    stat = path.stat()

    return {
        "path": str(path),
        "size_bytes": int(stat.st_size),
        "mtime_ns": int(stat.st_mtime_ns),
    }


def cache_marker_path(
    output_h5ad: Path,
) -> Path:
    return output_h5ad.with_name(
        f"{output_h5ad.name}.{CACHE_VERSION}.json"
    )


def make_core_mask_if_requested(
    obs: pd.DataFrame,
) -> pd.Series:
    if not FILTER_TO_CORE_CANDIDATES_IF_PRESENT:
        return pd.Series(
            True,
            index=obs.index,
            dtype=bool,
        )

    candidate_columns = [
        "strict_core_gbm_idhwt_candidate",
        "core_gbm_idhwt_candidate",
        "is_core",
        "core_candidate",
    ]

    for column in candidate_columns:
        if column in obs.columns:
            values = (
                obs[column]
                .astype(str)
                .str.strip()
                .str.lower()
            )
            return values.isin(
                ["yes", "true", "1", "y", "core"]
            )

    if "analysis_group_major" in obs.columns:
        values = (
            obs["analysis_group_major"]
            .astype(str)
            .str.strip()
            .str.lower()
        )
        return values.eq("core")

    print(
        "[WARNING] Core filtering was requested, but no "
        "core-candidate column was found. Keeping all cells."
    )
    return pd.Series(
        True,
        index=obs.index,
        dtype=bool,
    )


In [4]:
# =========================
# 3) GENCODE Ensembl-to-symbol mapping
# =========================

GENCODE_MAP_TSV = (
    METADATA_DIR
    / "gencode_v49_ensembl_to_gene_symbol_mapping.tsv"
)


def parse_gtf_attributes(
    attribute_text: str,
) -> dict:
    output = {}

    for item in attribute_text.strip().split(";"):
        item = item.strip()

        if not item or " " not in item:
            continue

        key, value = item.split(" ", 1)
        output[key] = value.strip().strip('"')

    return output


def build_or_load_gencode_map(
    gtf_path: Path,
    cache_tsv: Path,
):
    if cache_tsv.exists():
        mapping = pd.read_csv(
            cache_tsv,
            sep="\t",
        )
        print(
            "[USING cached GENCODE map]",
            cache_tsv,
            "shape=",
            mapping.shape,
        )

    else:
        if not gtf_path.exists():
            raise FileNotFoundError(
                f"GENCODE GTF not found: {gtf_path}\n"
                "This file is required to convert DS1/DS2 "
                "Ensembl IDs to gene symbols."
            )

        rows = []
        opener = (
            gzip.open
            if str(gtf_path).endswith(".gz")
            else open
        )

        with opener(
            gtf_path,
            "rt",
        ) as handle:
            for line in handle:
                if not line or line.startswith("#"):
                    continue

                parts = line.rstrip("\n").split("\t")
                if len(parts) < 9 or parts[2] != "gene":
                    continue

                (
                    chromosome,
                    source,
                    feature,
                    start,
                    end,
                    score,
                    strand,
                    frame,
                    attributes,
                ) = parts

                parsed = parse_gtf_attributes(
                    attributes
                )

                gene_id = parsed.get(
                    "gene_id",
                    "",
                )
                gene_name = parsed.get(
                    "gene_name",
                    "",
                )
                gene_type = parsed.get(
                    "gene_type",
                    parsed.get(
                        "gene_biotype",
                        "",
                    ),
                )

                if not gene_id or not gene_name:
                    continue

                rows.append(
                    {
                        "ensembl_id_raw": gene_id,
                        "ensembl_id": re.sub(
                            r"\.\d+$",
                            "",
                            gene_id,
                        ),
                        "gene_symbol": gene_name,
                        "gene_type": gene_type,
                        "chromosome": chromosome,
                        "start": int(start),
                        "end": int(end),
                        "strand": strand,
                    }
                )

        mapping = (
            pd.DataFrame(rows)
            .drop_duplicates(
                subset=["ensembl_id"],
                keep="first",
            )
        )
        write_tsv_atomic(
            mapping,
            cache_tsv,
        )

        print(
            "[BUILT GENCODE map]",
            cache_tsv,
            "shape=",
            mapping.shape,
        )

    required_columns = {
        "ensembl_id",
        "gene_symbol",
    }
    missing = (
        required_columns
        - set(mapping.columns)
    )

    if missing:
        raise ValueError(
            "GENCODE mapping table is missing columns: "
            f"{sorted(missing)}"
        )

    map_dict = dict(
        zip(
            mapping["ensembl_id"].astype(str),
            mapping["gene_symbol"].astype(str),
        )
    )

    return mapping, map_dict


gencode_map, ensembl_to_symbol = (
    build_or_load_gencode_map(
        GTF_PATH,
        GENCODE_MAP_TSV,
    )
)

print("GENCODE mapping rows:", len(gencode_map))
display(gencode_map.head())


[USING cached GENCODE map] G:\Repository\glioma-cnv-single-cell-atlas\metadata\gencode_v49_ensembl_to_gene_symbol_mapping.tsv shape= (78899, 8)
GENCODE mapping rows: 78899


,ensembl_id_raw,ensembl_id,gene_symbol,gene_type,chromosome,start,end,strand
0,ENSG00000290825.2,ENSG00000290825,DDX11L16,lncRNA,chr1,11121,24894,+
1,ENSG00000223972.6,ENSG00000223972,DDX11L1,transcribed_unprocessed_pseudogene,chr1,12010,13670,+
2,ENSG00000310526.1,ENSG00000310526,WASH7P,lncRNA,chr1,14356,30744,-
3,ENSG00000227232.6,ENSG00000227232,WASH7P,transcribed_unprocessed_pseudogene,chr1,14696,24886,-
4,ENSG00000278267.1,ENSG00000278267,MIR6859-1,miRNA,chr1,17369,17436,-


In [5]:
# =========================
# 4) Direct H5AD / CSR helpers
# =========================

def decode_h5_attribute(value):
    if isinstance(value, bytes):
        return value.decode("utf-8")

    return value


def stored_matrix_shape(node) -> tuple[int, int]:
    if isinstance(node, h5py.Dataset):
        return tuple(
            int(value)
            for value in node.shape
        )

    if not isinstance(node, h5py.Group):
        raise TypeError(
            "Unsupported matrix node: "
            f"{type(node).__name__}"
        )

    shape = node.attrs.get("shape")
    if shape is None:
        raise KeyError(
            f"Sparse matrix group {node.name!r} "
            "does not contain a shape attribute."
        )

    return tuple(int(value) for value in shape)


def stored_sparse_encoding(
    group: h5py.Group,
) -> str:
    return str(
        decode_h5_attribute(
            group.attrs.get(
                "encoding-type",
                "",
            )
        )
    )


def require_csr_group(
    group: h5py.Group,
    label: str,
):
    encoding = stored_sparse_encoding(group)

    if encoding != "csr_matrix":
        raise RuntimeError(
            f"{label} must be stored as CSR for the "
            "memory-safe row reader; observed "
            f"{encoding!r}."
        )

    required = {
        "data",
        "indices",
        "indptr",
    }
    missing = required - set(group.keys())

    if missing:
        raise KeyError(
            f"{label} is missing CSR arrays: "
            f"{sorted(missing)}"
        )


def read_csr_rows(
    group: h5py.Group,
    row_start: int,
    row_end: int,
) -> sparse.csr_matrix:
    require_csr_group(
        group,
        group.name,
    )

    n_rows, n_columns = stored_matrix_shape(
        group
    )

    if not (
        0 <= row_start <= row_end <= n_rows
    ):
        raise IndexError(
            "Invalid CSR row interval: "
            f"[{row_start}, {row_end}) for "
            f"{n_rows} rows."
        )

    global_indptr = np.asarray(
        group["indptr"][
            row_start : row_end + 1
        ],
        dtype=np.int64,
    )

    data_start = int(global_indptr[0])
    data_end = int(global_indptr[-1])

    local_indptr = (
        global_indptr - data_start
    )

    data = np.asarray(
        group["data"][data_start:data_end]
    )
    indices = np.asarray(
        group["indices"][data_start:data_end],
        dtype=np.int32,
    )

    matrix = sparse.csr_matrix(
        (
            data,
            indices,
            local_indptr,
        ),
        shape=(
            row_end - row_start,
            n_columns,
        ),
        dtype=data.dtype,
    )

    matrix.sort_indices()
    return matrix


def validate_raw_count_chunk(
    matrix: sparse.csr_matrix,
    label: str,
):
    values = matrix.data

    if values.size == 0:
        return

    if not np.isfinite(values).all():
        raise ValueError(
            f"{label} contains non-finite values."
        )

    if np.any(values < 0):
        raise ValueError(
            f"{label} contains negative values."
        )

    if not np.equal(
        values,
        np.floor(values),
    ).all():
        raise ValueError(
            f"{label} contains non-integer values."
        )


def read_h5ad_metadata(
    path: Path,
):
    path = Path(path)

    with h5py.File(
        path,
        "r",
    ) as handle:
        if "X" not in handle:
            raise KeyError(
                f"{path} is missing /X."
            )

        if "obs" not in handle or "var" not in handle:
            raise KeyError(
                f"{path} is missing /obs or /var."
            )

        shape = stored_matrix_shape(
            handle["X"]
        )
        obs = read_elem(handle["obs"])
        var = read_elem(handle["var"])

        layers = (
            list(handle["layers"].keys())
            if "layers" in handle
            else []
        )

        counts_shape = None
        counts_encoding = None
        counts_nnz = None

        if (
            "layers" in handle
            and "counts" in handle["layers"]
        ):
            counts_node = (
                handle["layers"]["counts"]
            )
            counts_shape = stored_matrix_shape(
                counts_node
            )

            if isinstance(
                counts_node,
                h5py.Group,
            ):
                counts_encoding = (
                    stored_sparse_encoding(
                        counts_node
                    )
                )
                if "data" in counts_node:
                    counts_nnz = int(
                        counts_node["data"].shape[0]
                    )

        x_encoding = (
            stored_sparse_encoding(handle["X"])
            if isinstance(
                handle["X"],
                h5py.Group,
            )
            else "dense"
        )

        x_nnz = (
            int(handle["X"]["data"].shape[0])
            if (
                isinstance(
                    handle["X"],
                    h5py.Group,
                )
                and "data" in handle["X"]
            )
            else None
        )

    return {
        "shape": shape,
        "obs": obs,
        "var": var,
        "layers": layers,
        "x_encoding": x_encoding,
        "x_nnz": x_nnz,
        "counts_shape": counts_shape,
        "counts_encoding": counts_encoding,
        "counts_nnz": counts_nnz,
    }


def validate_h5ad_structure(
    path: Path,
    expected_shape=None,
    expected_var_names=None,
):
    metadata = read_h5ad_metadata(path)

    if (
        expected_shape is not None
        and tuple(metadata["shape"])
        != tuple(expected_shape)
    ):
        raise AssertionError(
            f"{path}: expected shape "
            f"{tuple(expected_shape)}, observed "
            f"{metadata['shape']}."
        )

    if "counts" not in metadata["layers"]:
        raise AssertionError(
            f"{path}: layers['counts'] is missing."
        )

    if (
        metadata["counts_shape"]
        != metadata["shape"]
    ):
        raise AssertionError(
            f"{path}: X and counts shapes differ."
        )

    if metadata["x_encoding"] != "csr_matrix":
        raise AssertionError(
            f"{path}: X is not stored as CSR; "
            f"observed {metadata['x_encoding']!r}."
        )

    if (
        metadata["counts_encoding"]
        != "csr_matrix"
    ):
        raise AssertionError(
            f"{path}: counts is not stored as CSR; "
            f"observed "
            f"{metadata['counts_encoding']!r}."
        )

    if (
        expected_var_names is not None
        and list(
            metadata["var"].index.astype(str)
        )
        != list(map(str, expected_var_names))
    ):
        raise AssertionError(
            f"{path}: var_names do not match the "
            "expected order."
        )

    if not metadata["obs"].index.is_unique:
        raise AssertionError(
            f"{path}: obs_names are not unique."
        )

    if not metadata["var"].index.is_unique:
        raise AssertionError(
            f"{path}: var_names are not unique."
        )

    return metadata


def concat_on_disk_compatible(
    input_files,
    output_file: Path,
):
    """
    Run concat_on_disk in a short-lived child Python process.

    On Windows, some AnnData/HDF5 combinations keep the new H5AD file
    open after concat_on_disk returns. Running concatenation in a child
    process guarantees that all HDF5 handles are closed when the child
    exits, before the parent reopens the file with mode='r+'.
    """
    output_file = Path(output_file)
    remove_path(output_file)

    function = ad.experimental.concat_on_disk
    signature = inspect.signature(function)
    supported = set(signature.parameters)

    optional = {
        "axis": 0,
        "join": "inner",
        "merge": "same",
        "uns_merge": "same",
        "index_unique": None,
        "max_loaded_elems": (
            CONCAT_MAX_LOADED_ELEMS
        ),
    }
    kwargs = {
        key: value
        for key, value in optional.items()
        if key in supported
    }

    config_path = output_file.with_name(
        f"{output_file.name}.concat_config.json"
    )
    remove_path(config_path)

    payload = {
        "in_files": [
            str(Path(path).resolve())
            for path in input_files
        ],
        "out_file": str(output_file.resolve()),
        "kwargs": kwargs,
    }
    write_json_atomic(payload, config_path)

    child_script = r"""
import inspect
import json
import sys
from pathlib import Path

import anndata as ad

config_path = Path(sys.argv[1])

with config_path.open("r", encoding="utf-8") as handle:
    payload = json.load(handle)

function = ad.experimental.concat_on_disk
supported = set(inspect.signature(function).parameters)
kwargs = {
    key: value
    for key, value in payload["kwargs"].items()
    if key in supported
}

print(
    "[child concat_on_disk]",
    len(payload["in_files"]),
    "files ->",
    payload["out_file"],
)
print("[child kwargs]", kwargs)

function(
    in_files=payload["in_files"],
    out_file=payload["out_file"],
    **kwargs,
)
"""

    print(
        "[concat_on_disk subprocess]",
        len(input_files),
        "files ->",
        output_file,
    )
    print("[concat_on_disk kwargs]", kwargs)

    environment = os.environ.copy()
    environment["PYTHONUNBUFFERED"] = "1"

    try:
        completed = subprocess.run(
            [
                sys.executable,
                "-c",
                child_script,
                str(config_path),
            ],
            cwd=str(PROJECT_DIR),
            env=environment,
            text=True,
            capture_output=True,
            check=False,
        )

        if completed.stdout:
            print(completed.stdout, end="")

        if completed.returncode != 0:
            if completed.stderr:
                print(
                    completed.stderr,
                    file=sys.stderr,
                    end="",
                )

            remove_path(output_file)
            raise RuntimeError(
                "anndata.experimental.concat_on_disk failed in the "
                f"child process with exit code {completed.returncode}. "
                "No in-memory fallback was used."
            )

        if not output_file.exists():
            raise FileNotFoundError(
                "concat_on_disk returned successfully but did not create "
                f"the expected output: {output_file}"
            )

    finally:
        remove_path(config_path)

    gc.collect()
    time.sleep(0.25)


def open_h5py_with_retry(
    path: Path,
    mode: str,
    attempts: int = 12,
    initial_delay_seconds: float = 0.25,
):
    path = Path(path)
    delay = float(initial_delay_seconds)
    last_error = None

    for attempt in range(1, attempts + 1):
        try:
            return h5py.File(path, mode)

        except PermissionError as error:
            last_error = error
            gc.collect()

            if attempt == attempts:
                break

            print(
                "[Windows HDF5 lock retry]",
                f"attempt {attempt}/{attempts}",
                path.name,
                f"sleep={delay:.2f}s",
            )
            time.sleep(delay)
            delay = min(delay * 1.5, 2.0)

    raise PermissionError(
        "Windows did not release the HDF5 file lock after "
        f"{attempts} attempts: {path}"
    ) from last_error


def replace_uns_on_disk(
    target_h5ad: Path,
    uns_mapping: dict,
):
    target_h5ad = Path(target_h5ad)
    sidecar = target_h5ad.with_name(
        f"{target_h5ad.name}.uns_sidecar.h5ad"
    )
    remove_path(sidecar)

    holder = ad.AnnData(
        uns=deepcopy(uns_mapping)
    )
    holder.write_h5ad(
        sidecar,
        compression="lzf",
    )

    del holder
    gc.collect()

    source = None
    target = None

    try:
        source = open_h5py_with_retry(
            sidecar,
            "r",
        )

        if "uns" not in source:
            raise AssertionError(
                "The metadata sidecar does not contain /uns."
            )

        target = open_h5py_with_retry(
            target_h5ad,
            "r+",
        )

        if "uns" in target:
            del target["uns"]

        source.copy(
            "uns",
            target,
            name="uns",
        )
        target.flush()

    finally:
        if target is not None:
            target.close()

        if source is not None:
            source.close()

        gc.collect()
        time.sleep(0.1)
        remove_path(sidecar)



def patch_uns_in_child_process(
    target_h5ad: Path,
    uns_mapping: dict,
):
    """
    Patch /uns into a completed H5AD in a separate Python process.

    This avoids the Windows/HDF5 file-lock problem observed when the parent
    process tries to reopen a concat_on_disk output in r+ mode.
    """
    target_h5ad = Path(target_h5ad).resolve()

    sidecar = (
        STAGE_DIR
        / f"{target_h5ad.stem}.{CACHE_VERSION}.uns_sidecar.h5ad"
    )
    remove_path(sidecar)

    holder = ad.AnnData(
        uns=deepcopy(uns_mapping)
    )
    holder.write_h5ad(
        sidecar,
        compression="lzf",
    )

    del holder
    gc.collect()

    child_script = r"""
import sys
import h5py

target_h5ad = sys.argv[1]
sidecar = sys.argv[2]

with h5py.File(sidecar, "r") as source:
    if "uns" not in source:
        raise RuntimeError("UNS sidecar does not contain /uns.")

    with h5py.File(target_h5ad, "r+") as target:
        if "uns" in target:
            del target["uns"]

        source.copy(
            "uns",
            target,
            name="uns",
        )
        target.flush()
"""

    environment = os.environ.copy()
    environment["PYTHONUNBUFFERED"] = "1"

    try:
        completed = subprocess.run(
            [
                sys.executable,
                "-c",
                child_script,
                str(target_h5ad),
                str(sidecar),
            ],
            cwd=str(PROJECT_DIR),
            env=environment,
            text=True,
            capture_output=True,
            check=False,
        )

        if completed.stdout:
            print(completed.stdout, end="")

        if completed.returncode != 0:
            if completed.stderr:
                print(
                    completed.stderr,
                    file=sys.stderr,
                    end="",
                )

            raise RuntimeError(
                "The child-process /uns patch failed with exit code "
                f"{completed.returncode}."
            )

    finally:
        remove_path(sidecar)

    gc.collect()
    time.sleep(0.25)


In [6]:
# =========================
# 5) Inspect the four post-QC inputs without loading matrices
# =========================

inspection_rows = []

for item in CORE_DATASETS:
    path = Path(item["path"])

    print("\n" + "=" * 100)
    print(item["core_dataset"])
    print(path)
    print("exists:", path.exists())

    row = {
        "core_dataset": item["core_dataset"],
        "dataset_id_config": item["dataset_id"],
        "geo_id_config": item["geo_id"],
        "path": str(path),
        "exists": path.exists(),
    }

    if not path.exists():
        inspection_rows.append(row)
        continue

    metadata = read_h5ad_metadata(path)
    obs = metadata["obs"]
    var = metadata["var"]

    row.update(
        {
            "n_obs": int(metadata["shape"][0]),
            "n_vars": int(metadata["shape"][1]),
            "layers": ",".join(
                metadata["layers"]
            ),
            "has_counts_layer": (
                "counts" in metadata["layers"]
            ),
            "x_encoding": (
                metadata["x_encoding"]
            ),
            "counts_encoding": (
                metadata["counts_encoding"]
            ),
            "counts_nnz": (
                metadata["counts_nnz"]
            ),
            "n_obs_columns": len(obs.columns),
            "n_var_columns": len(var.columns),
            "duplicated_var_names": int(
                var.index.duplicated().sum()
            ),
            "first_10_var_names": ";".join(
                map(str, var.index[:10])
            ),
            "var_names_ensembl_like": (
                is_ensembl_like(var.index)
            ),
            "obs_columns": ";".join(
                map(str, obs.columns)
            ),
            "var_columns": ";".join(
                map(str, var.columns)
            ),
        }
    )

    print(
        "shape:",
        metadata["shape"],
    )
    print(
        "layers:",
        metadata["layers"],
    )
    print(
        "counts encoding:",
        metadata["counts_encoding"],
    )
    print(
        "first 10 var_names:",
        list(var.index[:10]),
    )

    if "counts" not in metadata["layers"]:
        raise ValueError(
            f"{path} is missing "
            "layers['counts']."
        )

    if (
        metadata["counts_encoding"]
        != "csr_matrix"
    ):
        raise ValueError(
            f"{path}: counts must be stored as CSR "
            "for memory-safe row processing."
        )

    if metadata["shape"][1] < MIN_GENES_PER_INPUT:
        raise ValueError(
            f"{path} has only "
            f"{metadata['shape'][1]} genes. "
            "This looks like an HVG/subset object."
        )

    inspection_rows.append(row)

    del metadata, obs, var
    gc.collect()


input_inspection = pd.DataFrame(
    inspection_rows
)

if not input_inspection["exists"].all():
    missing_paths = input_inspection.loc[
        ~input_inspection["exists"],
        "path",
    ].tolist()
    raise FileNotFoundError(
        f"Missing post-QC inputs: {missing_paths}"
    )

write_tsv_atomic(
    input_inspection,
    INPUT_INSPECTION_TSV,
)
display(input_inspection)



DS1_GSE103224
G:\Repository\glioma-cnv-single-cell-atlas\data\processed\DS1_GSE103224\GSE103224_postQC.h5ad
exists: True
shape: (23207, 29457)
layers: ['counts']
counts encoding: csr_matrix
first 10 var_names: ['ENSG00000130876', 'ENSG00000269696', 'ENSG00000230393', 'ENSG00000165244', 'ENSG00000173597', 'ENSG00000158022', 'ENSG00000213018', 'ENSG00000167220', 'ENSG00000231154', 'ENSG00000240519']

DS2_GSE141383
G:\Repository\glioma-cnv-single-cell-atlas\data\processed\DS2_GSE141383\GSE141383_postQC.h5ad
exists: True
shape: (21391, 25835)
layers: ['counts']
counts encoding: csr_matrix
first 10 var_names: ['RP11-1072C15.4', 'BET1', 'LINC01198', 'ICE2', 'THRB-AS1', 'CTSH', 'RP11-755E23.2', 'ZNF106', 'TSR1', 'BHLHB9']

DS3_GSE182109
G:\Repository\glioma-cnv-single-cell-atlas\data\processed\DS3_GSE182109\GSE182109_postQC.h5ad
exists: True
shape: (218735, 30284)
layers: ['counts']
counts encoding: csr_matrix
first 10 var_names: ['A1BG', 'A1BG-AS1', 'A1CF', 'A2M', 'A2M-AS1', 'A2ML1', 'A2ML1

,core_dataset,dataset_id_config,geo_id_config,path,exists,n_obs,n_vars,layers,has_counts_layer,x_encoding,counts_encoding,counts_nnz,n_obs_columns,n_var_columns,duplicated_var_names,first_10_var_names,var_names_ensembl_like,obs_columns,var_columns
0,DS1_GSE103224,DS1,GSE103224,G:\Repository\glioma-cnv-single-cell-atlas\dat...,True,23207,29457,counts,True,csr_matrix,csr_matrix,30643240,22,12,0,ENSG00000130876;ENSG00000269696;ENSG0000023039...,True,sample_id;batch;geo_sample_id;geo_id;core_data...,ensembl_id_version;gene_symbol;mt;ribo;hb;n_ce...
1,DS2_GSE141383,DS2,GSE141383,G:\Repository\glioma-cnv-single-cell-atlas\dat...,True,21391,25835,counts,True,csr_matrix,csr_matrix,24443874,25,12,0,RP11-1072C15.4;BET1;LINC01198;ICE2;THRB-AS1;CT...,False,cell_barcode;sample;sample_id;batch;geo_sample...,gene_symbols;mt;ribo;hb;n_cells_by_counts;mean...
2,DS3_GSE182109,DS3,GSE182109,G:\Repository\glioma-cnv-single-cell-atlas\dat...,True,218735,30284,counts,True,csr_matrix,csr_matrix,572656313,24,13,0,A1BG;A1BG-AS1;A1CF;A2M;A2M-AS1;A2ML1;A2ML1-AS1...,False,cell_barcode;gsm;sample;sample_id;group;patien...,gene_ids;feature_types;feature_name_original;m...
3,DS4_GSE278450,DS4,GSE278450,G:\Repository\glioma-cnv-single-cell-atlas\dat...,True,88094,24584,counts,True,csr_matrix,csr_matrix,182316786,15,9,0,RP11-34P13.7;RP11-34P13.8;FO538757.2;AP006222....,False,cell_barcode;sample_id;sample;geo_id;core_data...,feature_name;mt;ribo;hb;n_cells_by_counts;mean...


In [7]:
# =========================
# 6) Gene mapping and observation-metadata helpers
# =========================

def cleaned_symbol_series(values) -> pd.Series:
    output = pd.Series(
        values,
        dtype="object",
    )

    output = output.map(
        lambda value: (
            None
            if pd.isna(value)
            else str(value).strip()
        )
    )

    return output


def ensembl_like_fraction(values) -> float:
    series = pd.Series(values).astype(str).str.strip()

    if len(series) == 0:
        return 0.0

    stripped = strip_ensembl_version(series)

    return float(
        stripped.str.match(
            r"^ENSG\d+$",
            na=False,
        ).mean()
    )


def resolve_ensembl_identifier_source(
    var: pd.DataFrame,
):
    """
    Resolve the original Ensembl identifiers used for GENCODE mapping.

    Historical DS1/DS2 analysis mapped Ensembl IDs from the feature index.
    The rebuilt DS2 object may instead expose gene symbols as var_names while
    retaining the original IDs in var['ensembl_id_version']; this helper
    restores the historical mapping semantics.
    """
    var_names = pd.Series(
        var.index.astype(str),
        dtype="object",
    )

    candidates = [
        (
            "var_names",
            var_names,
        ),
    ]

    for column in (
        "ensembl_id_version",
        "ensembl_id",
        "gene_id",
        "gene_ids",
    ):
        if column in var.columns:
            candidates.append(
                (
                    f"var[{column}]",
                    pd.Series(
                        var[column].to_numpy(),
                        dtype="object",
                    ),
                )
            )

    scored = []

    for label, values in candidates:
        fraction = ensembl_like_fraction(values)
        scored.append(
            (
                fraction,
                label,
                values,
            )
        )

    scored.sort(
        key=lambda item: item[0],
        reverse=True,
    )

    best_fraction, best_label, best_values = scored[0]

    if best_fraction < 0.50:
        return None, None, best_fraction

    return (
        best_values.reset_index(drop=True),
        best_label,
        best_fraction,
    )


def choose_gene_symbols_from_var(
    var: pd.DataFrame,
    item: dict,
    mapping_dict: dict,
):
    var_names = pd.Index(
        var.index.astype(str),
        dtype="object",
    )
    original = pd.Series(
        var_names.to_numpy(),
        dtype="object",
    )

    mode = item.get(
        "gene_id_mode",
        "auto",
    )

    if mode == "ensembl":
        (
            ensembl_identifiers,
            ensembl_identifier_source,
            ensembl_like_frac,
        ) = resolve_ensembl_identifier_source(var)

        if ensembl_identifiers is None:
            raise RuntimeError(
                f"{item['core_dataset']}: configured for Ensembl mapping, "
                "but no Ensembl-like identifier source could be resolved "
                "from var_names or Ensembl metadata columns. "
                f"Best Ensembl-like fraction={ensembl_like_frac:.3f}."
            )

        stripped = strip_ensembl_version(
            ensembl_identifiers
        )
        symbols = stripped.map(mapping_dict)
        mapped_fraction = float(
            symbols.notna().mean()
        )

        print(
            "Ensembl identifier source:",
            ensembl_identifier_source,
        )
        print(
            "Ensembl-like fraction:",
            ensembl_like_frac,
        )
        print(
            "GENCODE v49 mapped fraction:",
            mapped_fraction,
        )

        if mapped_fraction < 0.50:
            raise RuntimeError(
                f"{item['core_dataset']}: GENCODE v49 mapped only "
                f"{mapped_fraction:.3f} of the resolved Ensembl IDs from "
                f"{ensembl_identifier_source}. Refusing to fall back to "
                "gene-symbol columns because that would change the historical "
                "DS1/DS2 harmonization semantics."
            )

        return (
            cleaned_symbol_series(symbols),
            "gencode_v49_ensembl_to_gene_symbol",
            mapped_fraction,
            ensembl_identifier_source,
        )

    if (
        mode == "auto"
        and is_ensembl_like(original)
    ):
        stripped = strip_ensembl_version(
            original
        )
        symbols = stripped.map(mapping_dict)
        mapped_fraction = float(
            symbols.notna().mean()
        )

        return (
            cleaned_symbol_series(symbols),
            "gencode_v49_ensembl_to_gene_symbol",
            mapped_fraction,
            "var_names",
        )

    if (
        mode == "symbol"
        or not is_ensembl_like(original)
    ):
        return (
            cleaned_symbol_series(original),
            "var_names_gene_symbol",
            np.nan,
            "var_names",
        )

    candidate_columns = [
        "gene_symbol",
        "gene_symbols",
        "feature_name",
        "gene_name",
        "symbol",
    ]

    for column in candidate_columns:
        if column in var.columns:
            return (
                cleaned_symbol_series(
                    var[column].to_numpy()
                ),
                f"var[{column}]",
                np.nan,
                f"var[{column}]",
            )

    return (
        cleaned_symbol_series(original),
        "var_names",
        np.nan,
        "var_names",
    )


def build_gene_collapse_plan(
    var: pd.DataFrame,
    symbols: pd.Series,
):
    original_var_names = pd.Series(
        var.index.astype(str),
        dtype="object",
    )
    symbols = cleaned_symbol_series(symbols)

    lower = (
        symbols.fillna("")
        .astype(str)
        .str.lower()
    )

    invalid = (
        symbols.isna()
        | lower.eq("")
        | lower.isin(
            [
                "nan",
                "none",
                "null",
                "na",
                "n/a",
                "<na>",
            ]
        )
        | symbols.fillna("")
        .astype(str)
        .str.startswith("ENSG")
    )

    valid_positions = np.flatnonzero(
        (~invalid).to_numpy()
    ).astype(np.int32)

    symbols_valid = (
        symbols.iloc[valid_positions]
        .astype(str)
        .reset_index(drop=True)
    )
    original_valid = (
        original_var_names
        .iloc[valid_positions]
        .astype(str)
        .reset_index(drop=True)
    )

    group_ids, unique_symbols = pd.factorize(
        symbols_valid,
        sort=False,
    )
    group_ids = group_ids.astype(
        np.int32,
        copy=False,
    )
    unique_symbols = pd.Index(
        unique_symbols.astype(str),
        dtype="object",
    )

    collapse_table = pd.DataFrame(
        {
            "gene_symbol": symbols_valid,
            "original_var_name": original_valid,
        }
    )

    output_var = (
        collapse_table
        .groupby(
            "gene_symbol",
            sort=False,
        )
        .agg(
            n_original_features_collapsed=(
                "original_var_name",
                "size",
            ),
            original_var_names_joined=(
                "original_var_name",
                lambda values: (
                    ";".join(
                        map(
                            str,
                            values.iloc[:20],
                        )
                    )
                    + (
                        ";..."
                        if len(values) > 20
                        else ""
                    )
                ),
            ),
        )
    )

    output_var.index = pd.Index(
        output_var.index.astype(str),
        dtype="object",
    )

    if not output_var.index.is_unique:
        raise AssertionError(
            "Collapsed gene-symbol index is not unique."
        )

    return {
        "valid_positions": valid_positions,
        "group_ids": group_ids,
        "unique_symbols": unique_symbols,
        "output_var": output_var,
        "n_invalid": int(invalid.sum()),
        "n_valid_features": int(
            len(valid_positions)
        ),
        "n_unique_symbols": int(
            len(unique_symbols)
        ),
    }


def standardize_obs_metadata(
    obs: pd.DataFrame,
    item: dict,
) -> pd.DataFrame:
    output = obs.copy()

    core_dataset = item["core_dataset"]
    dataset_id = item["dataset_id"]
    geo_id = item["geo_id"]

    output["core_dataset"] = core_dataset
    output["dataset_id"] = dataset_id
    output["geo_id"] = geo_id
    output["study_id"] = geo_id
    output["source_dataset"] = core_dataset
    output["source_file"] = str(item["path"])
    output["concat_key"] = core_dataset

    if "sample_id" not in output.columns:
        for column in (
            "sample",
            "sample_name",
            "orig.ident",
            "library_id",
        ):
            if column in output.columns:
                output["sample_id"] = (
                    output[column].astype(str)
                )
                break
        else:
            output["sample_id"] = (
                f"{core_dataset}_sample_unknown"
            )

    output["sample_id"] = clean_string_series(
        output["sample_id"],
        f"{core_dataset}_sample_unknown",
    ).to_numpy(dtype=object)

    if "patient_id" not in output.columns:
        for column in (
            "patient",
            "patient_name",
            "donor",
            "donor_id",
            "case_id",
        ):
            if column in output.columns:
                output["patient_id"] = (
                    output[column].astype(str)
                )
                break
        else:
            output["patient_id"] = (
                output["sample_id"].astype(str)
            )

    output["patient_id"] = clean_string_series(
        output["patient_id"],
        f"{core_dataset}_patient_unknown",
    ).to_numpy(dtype=object)

    if "condition" not in output.columns:
        for column in (
            "diagnosis",
            "disease",
            "group",
        ):
            if column in output.columns:
                output["condition"] = (
                    output[column].astype(str)
                )
                break
        else:
            output["condition"] = "GBM"

    output["condition"] = clean_string_series(
        output["condition"],
        "GBM",
    ).to_numpy(dtype=object)

    output["batch_dataset"] = (
        output["core_dataset"]
        .astype(str)
        .to_numpy(dtype=object)
    )
    output["batch_sample"] = (
        output["sample_id"]
        .astype(str)
        .to_numpy(dtype=object)
    )
    output["batch_patient"] = (
        output["patient_id"]
        .astype(str)
        .to_numpy(dtype=object)
    )

    original_names = pd.Index(
        output.index.astype(str),
        dtype="object",
    )
    output["original_cell_barcode"] = (
        original_names.to_numpy(
            dtype=object,
            copy=True,
        )
    )

    prefixed = [
        f"{core_dataset}:{value}"
        for value in original_names
    ]
    output.index = make_unique_index(
        prefixed
    )
    output.index.name = "cell_id"

    return output


In [8]:
# =========================
# 7) Chunked H5AD construction helpers
# =========================

def write_expression_chunk(
    matrix: sparse.csr_matrix,
    obs_chunk: pd.DataFrame,
    var: pd.DataFrame,
    output_path: Path,
):
    output_path = Path(output_path)
    temporary = output_path.with_name(
        f"{output_path.name}.temporary"
    )

    remove_path(temporary)
    remove_path(output_path)

    matrix = matrix.tocsr()
    matrix.sum_duplicates()
    matrix.eliminate_zeros()
    matrix.sort_indices()

    chunk = ad.AnnData(
        X=matrix,
        obs=obs_chunk,
        var=var,
    )

    # AnnData stores the same sparse object in both locations without
    # creating a second Python-level matrix copy.
    chunk.layers["counts"] = chunk.X

    chunk.write_h5ad(
        temporary,
        compression="lzf",
    )
    os.replace(
        temporary,
        output_path,
    )

    del chunk
    gc.collect()


def output_cache_is_valid(
    output_h5ad: Path,
    marker_path: Path,
    expected_shape,
    expected_var_names,
    source_fingerprint: dict,
    operation: str,
) -> bool:
    output_h5ad = Path(output_h5ad)
    marker_path = Path(marker_path)

    if (
        not output_h5ad.exists()
        or not marker_path.exists()
    ):
        return False

    try:
        with marker_path.open(
            "r",
            encoding="utf-8",
        ) as handle:
            marker = json.load(handle)

        if (
            marker.get("cache_version")
            not in COMPATIBLE_CACHE_VERSIONS
        ):
            return False

        if marker.get("operation") != operation:
            return False

        if (
            marker.get("source_fingerprint")
            != source_fingerprint
        ):
            return False

        if (
            tuple(marker.get("shape", []))
            != tuple(expected_shape)
        ):
            return False

        if (
            marker.get("var_names_sha256")
            != strings_sha256(
                expected_var_names
            )
        ):
            return False

        validate_h5ad_structure(
            output_h5ad,
            expected_shape=expected_shape,
            expected_var_names=(
                expected_var_names
            ),
        )

        return True

    except Exception as error:
        print(
            "[CACHE INVALID]",
            output_h5ad,
            repr(error),
        )
        return False


def validate_uns_mapping(
    target_h5ad: Path,
    uns_mapping: dict,
):
    target_h5ad = Path(target_h5ad)

    with h5py.File(
        target_h5ad,
        "r",
    ) as handle:
        if "uns" not in handle:
            raise AssertionError(
                f"{target_h5ad}: /uns is missing."
            )

        saved_uns = read_elem(handle["uns"])

    missing_keys = sorted(
        set(uns_mapping) - set(saved_uns)
    )

    if missing_keys:
        raise AssertionError(
            f"{target_h5ad}: saved /uns is missing "
            f"top-level keys {missing_keys}."
        )


def finalize_chunk_files(
    chunk_files,
    output_h5ad: Path,
    expected_shape,
    expected_var_names,
    uns_mapping: dict,
):
    output_h5ad = Path(output_h5ad)

    if not chunk_files:
        raise ValueError(
            "At least one real expression chunk is required."
        )

    scratch_output = (
        STAGE_DIR
        / f"{output_h5ad.stem}.{CACHE_VERSION}.assembled.h5ad"
    )

    remove_path(scratch_output)

    try:
        concat_on_disk_compatible(
            chunk_files,
            scratch_output,
        )

        # Some AnnData releases drop /uns during concat_on_disk.
        # Patch provenance after concatenation in a separate child process.
        patch_uns_in_child_process(
            scratch_output,
            uns_mapping,
        )

        validate_h5ad_structure(
            scratch_output,
            expected_shape=expected_shape,
            expected_var_names=expected_var_names,
        )
        validate_uns_mapping(
            scratch_output,
            uns_mapping,
        )

        publish_file_cross_drive(
            scratch_output,
            output_h5ad,
        )

        validate_h5ad_structure(
            output_h5ad,
            expected_shape=expected_shape,
            expected_var_names=expected_var_names,
        )
        validate_uns_mapping(
            output_h5ad,
            uns_mapping,
        )

    finally:
        remove_path(scratch_output)


def build_harmonized_dataset(
    item: dict,
    mapping_dict: dict,
):
    input_path = Path(item["path"])
    output_h5ad = (
        HARM_DIR
        / (
            f"{item['core_dataset']}"
            "_postQC_gene_symbol_harmonized_counts.h5ad"
        )
    )
    marker_path = cache_marker_path(
        output_h5ad
    )

    metadata = read_h5ad_metadata(
        input_path
    )
    input_shape = metadata["shape"]
    input_obs = metadata["obs"]
    input_var = metadata["var"]

    if input_shape[1] < MIN_GENES_PER_INPUT:
        raise ValueError(
            f"{input_path} has only "
            f"{input_shape[1]} genes."
        )

    core_mask = make_core_mask_if_requested(
        input_obs
    )

    if not core_mask.all():
        raise RuntimeError(
            "The memory-safe notebook intentionally "
            "expects all post-QC cells. Set "
            "FILTER_TO_CORE_CANDIDATES_IF_PRESENT=False."
        )

    (
        symbols,
        symbol_source,
        mapped_fraction,
        ensembl_identifier_source,
    ) = choose_gene_symbols_from_var(
        input_var,
        item,
        mapping_dict,
    )

    plan = build_gene_collapse_plan(
        input_var,
        symbols,
    )
    output_var = plan["output_var"]
    output_var_names = pd.Index(
        output_var.index.astype(str),
        dtype="object",
    )

    historical_expected_genes = (
        HISTORICAL_HARMONIZED_GENE_COUNTS[
            item["core_dataset"]
        ]
    )

    if len(output_var_names) != historical_expected_genes:
        raise AssertionError(
            f"{item['core_dataset']}: historical harmonized gene count "
            f"is {historical_expected_genes}, but the current mapping plan "
            f"produces {len(output_var_names)}. "
            f"symbol_source={symbol_source!r}, "
            f"ensembl_identifier_source={ensembl_identifier_source!r}, "
            f"mapped_fraction={mapped_fraction!r}."
        )

    print(
        "Historical harmonized-gene count reproduced:",
        historical_expected_genes,
    )

    output_obs = standardize_obs_metadata(
        input_obs,
        item,
    )

    expected_shape = (
        input_shape[0],
        len(output_var_names),
    )
    source_fingerprint = file_fingerprint(
        input_path
    )

    if output_cache_is_valid(
        output_h5ad,
        marker_path,
        expected_shape,
        output_var_names,
        source_fingerprint,
        operation="gene_symbol_harmonization",
    ):
        print(
            "[REUSING valid harmonized cache]",
            output_h5ad,
        )

    else:
        print(
            "[BUILD harmonized]",
            item["core_dataset"],
            input_shape,
            "->",
            expected_shape,
        )

        remove_path(output_h5ad)
        remove_path(marker_path)

        chunk_dir = (
            WORK_DIR
            / (
                f"{item['core_dataset']}"
                "_harmonization_chunks"
            )
        )
        remove_path(chunk_dir)
        chunk_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        chunk_files = []

        with h5py.File(
            input_path,
            "r",
        ) as handle:
            counts_group = (
                handle["layers"]["counts"]
            )
            require_csr_group(
                counts_group,
                (
                    f"{item['core_dataset']} "
                    "layers/counts"
                ),
            )

            counts_dtype = (
                counts_group["data"].dtype
            )

            aggregation_matrix = sparse.csr_matrix(
                (
                    np.ones(
                        len(plan["valid_positions"]),
                        dtype=counts_dtype,
                    ),
                    (
                        plan["valid_positions"],
                        plan["group_ids"],
                    ),
                ),
                shape=(
                    input_shape[1],
                    len(output_var_names),
                ),
                dtype=counts_dtype,
            )

            for chunk_index, row_start in enumerate(
                range(
                    0,
                    input_shape[0],
                    ROW_CHUNK_SIZE,
                )
            ):
                row_end = min(
                    row_start + ROW_CHUNK_SIZE,
                    input_shape[0],
                )

                raw_chunk = read_csr_rows(
                    counts_group,
                    row_start,
                    row_end,
                )
                validate_raw_count_chunk(
                    raw_chunk,
                    (
                        f"{item['core_dataset']} "
                        f"rows {row_start}:{row_end}"
                    ),
                )

                collapsed = (
                    raw_chunk
                    @ aggregation_matrix
                ).tocsr()

                collapsed.sum_duplicates()
                collapsed.eliminate_zeros()
                collapsed.sort_indices()

                chunk_path = (
                    chunk_dir
                    / f"chunk_{chunk_index:06d}.h5ad"
                )

                write_expression_chunk(
                    collapsed,
                    output_obs.iloc[
                        row_start:row_end
                    ].copy(),
                    output_var,
                    chunk_path,
                )
                chunk_files.append(chunk_path)

                if (
                    chunk_index == 0
                    or (chunk_index + 1) % 25 == 0
                    or row_end == input_shape[0]
                ):
                    print(
                        f"  chunk {chunk_index + 1}: "
                        f"rows {row_start:,}:{row_end:,}"
                    )

                del raw_chunk, collapsed
                gc.collect()

            del aggregation_matrix

        if not chunk_files:
            raise RuntimeError(
                f"No chunks were created for "
                f"{item['core_dataset']}."
            )

        harmonization_uns = {
            "harmonization_provenance": {
                "cache_version": CACHE_VERSION,
                "core_dataset": (
                    item["core_dataset"]
                ),
                "source_h5ad": str(input_path),
                "gene_symbol_source": (
                    symbol_source
                ),
                "ensembl_identifier_source": (
                    ensembl_identifier_source
                ),
                "ensembl_mapped_fraction": (
                    "not_applicable"
                    if pd.isna(mapped_fraction)
                    else float(mapped_fraction)
                ),
                "duplicate_gene_symbols": (
                    "summed from raw counts"
                ),
                "invalid_or_unmapped_features_removed": int(
                    plan["n_invalid"]
                ),
                "raw_counts_storage": (
                    "X and layers/counts"
                ),
                "row_chunk_size": int(
                    ROW_CHUNK_SIZE
                ),
            }
        }

        finalize_chunk_files(
            chunk_files,
            output_h5ad,
            expected_shape,
            output_var_names,
            harmonization_uns,
        )

        write_json_atomic(
            {
                "cache_version": CACHE_VERSION,
                "operation": (
                    "gene_symbol_harmonization"
                ),
                "source_fingerprint": (
                    source_fingerprint
                ),
                "shape": list(expected_shape),
                "var_names_sha256": (
                    strings_sha256(
                        output_var_names
                    )
                ),
            },
            marker_path,
        )

        remove_path(chunk_dir)

    report = {
        "core_dataset": item["core_dataset"],
        "input_file": str(input_path),
        "n_cells_input": int(input_shape[0]),
        "n_genes_input": int(input_shape[1]),
        "n_invalid_or_unmapped_gene_symbols_removed": int(
            plan["n_invalid"]
        ),
        "n_features_after_invalid_removal": int(
            plan["n_valid_features"]
        ),
        "n_unique_gene_symbols_after_collapse": int(
            plan["n_unique_symbols"]
        ),
        "n_duplicate_features_collapsed": int(
            plan["n_valid_features"]
            - plan["n_unique_symbols"]
        ),
        "has_counts_layer_output": True,
        "symbol_source": symbol_source,
        "ensembl_identifier_source": (
            ensembl_identifier_source
        ),
        "ensembl_mapped_fraction": (
            mapped_fraction
        ),
        "n_samples": int(
            output_obs["sample_id"].nunique()
        ),
        "n_patients": int(
            output_obs["patient_id"].nunique()
        ),
        "conditions": " | ".join(
            sorted(
                output_obs["condition"]
                .astype(str)
                .unique()
            )
        ),
        "harmonized_h5ad": str(
            output_h5ad
        ),
    }

    del metadata, input_obs, input_var
    del output_obs, output_var, symbols, plan
    gc.collect()

    return (
        output_h5ad,
        set(map(str, output_var_names)),
        report,
    )


In [9]:
# =========================
# 8) Harmonize each dataset on disk
# =========================

harmonized_files = {}
gene_sets = {}
harmonization_rows = []

for item in CORE_DATASETS:
    print("\n" + "=" * 100)
    print(
        "[HARMONIZE]",
        item["core_dataset"],
    )

    (
        harmonized_path,
        gene_set,
        report,
    ) = build_harmonized_dataset(
        item,
        ensembl_to_symbol,
    )

    if len(gene_set) < MIN_GENES_PER_INPUT:
        raise ValueError(
            f"After harmonization, "
            f"{item['core_dataset']} has only "
            f"{len(gene_set)} gene symbols."
        )

    harmonized_files[
        item["core_dataset"]
    ] = harmonized_path
    gene_sets[
        item["core_dataset"]
    ] = gene_set
    harmonization_rows.append(report)

    print(
        "[HARMONIZED]",
        harmonized_path,
        "genes=",
        len(gene_set),
    )


gene_harmonization = pd.DataFrame(
    harmonization_rows
)
write_tsv_atomic(
    gene_harmonization,
    GENE_HARMONIZATION_TSV,
)
display(gene_harmonization)



[HARMONIZE] DS1_GSE103224
Ensembl identifier source: var_names
Ensembl-like fraction: 0.9999660522116984
GENCODE v49 mapped fraction: 0.9715517534032658
Historical harmonized-gene count reproduced: 23982
[BUILD harmonized] DS1_GSE103224 (23207, 29457) -> (23207, 23982)
  chunk 1: rows 0:1,024
  chunk 23: rows 22,528:23,207
[concat_on_disk subprocess] 23 files -> E:\glioma-cnv-scratch\main_analysis_01\stage\DS1_GSE103224_postQC_gene_symbol_harmonized_counts.main-analysis-01-nvme-scratch-v6-ds2-ensembl-source-fix.assembled.h5ad
[concat_on_disk kwargs] {'axis': 0, 'join': 'inner', 'merge': 'same', 'uns_merge': 'same', 'index_unique': None, 'max_loaded_elems': 3000000}
[child concat_on_disk] 23 files -> E:\glioma-cnv-scratch\main_analysis_01\stage\DS1_GSE103224_postQC_gene_symbol_harmonized_counts.main-analysis-01-nvme-scratch-v6-ds2-ensembl-source-fix.assembled.h5ad
[child kwargs] {'axis': 0, 'index_unique': None, 'join': 'inner', 'max_loaded_elems': 3000000, 'merge': 'same', 'uns_merge'

,core_dataset,input_file,n_cells_input,n_genes_input,n_invalid_or_unmapped_gene_symbols_removed,n_features_after_invalid_removal,n_unique_gene_symbols_after_collapse,n_duplicate_features_collapsed,has_counts_layer_output,symbol_source,ensembl_identifier_source,ensembl_mapped_fraction,n_samples,n_patients,conditions,harmonized_h5ad
0,DS1_GSE103224,G:\Repository\glioma-cnv-single-cell-atlas\dat...,23207,29457,5337,24120,23982,138,True,gencode_v49_ensembl_to_gene_symbol,var_names,0.971552,8,8,GBM,G:\Repository\glioma-cnv-single-cell-atlas\dat...
1,DS2_GSE141383,G:\Repository\glioma-cnv-single-cell-atlas\dat...,21391,25835,3956,21879,21840,39,True,gencode_v49_ensembl_to_gene_symbol,var[ensembl_id_version],0.975344,9,9,GBM,G:\Repository\glioma-cnv-single-cell-atlas\dat...
2,DS3_GSE182109,G:\Repository\glioma-cnv-single-cell-atlas\dat...,218735,30284,0,30284,30284,0,True,var_names_gene_symbol,var_names,NaN,44,11,LGG | ndGBM | rGBM,G:\Repository\glioma-cnv-single-cell-atlas\dat...
3,DS4_GSE278450,G:\Repository\glioma-cnv-single-cell-atlas\dat...,88094,24584,0,24584,24584,0,True,var_names_gene_symbol,var_names,NaN,22,22,GBM,G:\Repository\glioma-cnv-single-cell-atlas\dat...


In [10]:
# =========================
# 9) Shared genes across all four datasets
# =========================

dataset_keys = [
    item["core_dataset"]
    for item in CORE_DATASETS
]

pair_rows = []

for first, second in combinations(
    dataset_keys,
    2,
):
    pair_rows.append(
        {
            "dataset_1": first,
            "dataset_2": second,
            "n_genes_1": len(
                gene_sets[first]
            ),
            "n_genes_2": len(
                gene_sets[second]
            ),
            "pairwise_intersection": len(
                gene_sets[first].intersection(
                    gene_sets[second]
                )
            ),
        }
    )


pairwise = pd.DataFrame(pair_rows)

common_genes = sorted(
    set.intersection(
        *[
            gene_sets[key]
            for key in dataset_keys
        ]
    )
)

common_genes_df = pd.DataFrame(
    {
        "gene_symbol": common_genes,
    }
)

write_tsv_atomic(
    pairwise,
    PAIRWISE_INTERSECTION_TSV,
)
write_tsv_atomic(
    common_genes_df,
    COMMON_GENES_TSV,
)

print(
    "Common gene symbols across all four datasets:",
    len(common_genes),
)

if len(common_genes) != HISTORICAL_COMMON_GENE_COUNT:
    raise AssertionError(
        "Archived Ubuntu execution produced exactly "
        f"{HISTORICAL_COMMON_GENE_COUNT} common genes, but the current "
        f"reconstruction produced {len(common_genes)}."
    )

print(
    "Archived historical common-gene count reproduced:",
    HISTORICAL_COMMON_GENE_COUNT,
)
display(pairwise)

if (
    len(common_genes)
    < MIN_COMMON_GENES_AFTER_HARMONIZATION
):
    raise RuntimeError(
        "Too few common genes after harmonization: "
        f"{len(common_genes)}."
    )


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\gene_symbol_pairwise_intersections_4datasets_corrected.tsv shape= (6, 5)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\common_genes_after_harmonization_4datasets_corrected.tsv shape= (15176, 1)
Common gene symbols across all four datasets: 15176
Archived historical common-gene count reproduced: 15176


,dataset_1,dataset_2,n_genes_1,n_genes_2,pairwise_intersection
0,DS1_GSE103224,DS2_GSE141383,23982,21840,21523
1,DS1_GSE103224,DS3_GSE182109,23982,30284,17251
2,DS1_GSE103224,DS4_GSE278450,23982,24584,15994
3,DS2_GSE141383,DS3_GSE182109,21840,30284,16431
4,DS2_GSE141383,DS4_GSE278450,21840,24584,15361
5,DS3_GSE182109,DS4_GSE278450,30284,24584,18342


In [11]:
# =========================
# 10) Chunked common-gene subsetting
# =========================

def build_common_gene_subset(
    core_dataset: str,
    harmonized_h5ad: Path,
    common_gene_order,
):
    harmonized_h5ad = Path(
        harmonized_h5ad
    )
    output_h5ad = (
        COMMON_DIR
        / (
            f"{core_dataset}"
            "_postQC_common_genes_counts.h5ad"
        )
    )
    marker_path = cache_marker_path(
        output_h5ad
    )

    metadata = read_h5ad_metadata(
        harmonized_h5ad
    )
    input_shape = metadata["shape"]
    obs = metadata["obs"]
    var = metadata["var"]

    positions = var.index.astype(str).get_indexer(
        common_gene_order
    )

    missing_mask = positions < 0
    if missing_mask.any():
        missing = [
            common_gene_order[index]
            for index in np.flatnonzero(
                missing_mask
            )[:20]
        ]
        raise ValueError(
            f"{core_dataset} is missing common "
            f"genes unexpectedly, including {missing}."
        )

    output_var = var.iloc[
        positions
    ].copy()
    output_var.index = pd.Index(
        list(map(str, common_gene_order)),
        dtype="object",
    )

    expected_shape = (
        input_shape[0],
        len(common_gene_order),
    )
    source_fingerprint = file_fingerprint(
        harmonized_h5ad
    )

    if output_cache_is_valid(
        output_h5ad,
        marker_path,
        expected_shape,
        common_gene_order,
        source_fingerprint,
        operation="common_gene_subset",
    ):
        print(
            "[REUSING valid common-gene cache]",
            output_h5ad,
        )

    else:
        print(
            "[BUILD common subset]",
            core_dataset,
            input_shape,
            "->",
            expected_shape,
        )

        remove_path(output_h5ad)
        remove_path(marker_path)

        chunk_dir = (
            WORK_DIR
            / f"{core_dataset}_common_gene_chunks"
        )
        remove_path(chunk_dir)
        chunk_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        chunk_files = []

        with h5py.File(
            harmonized_h5ad,
            "r",
        ) as handle:
            counts_group = (
                handle["layers"]["counts"]
            )
            require_csr_group(
                counts_group,
                (
                    f"{core_dataset} "
                    "harmonized counts"
                ),
            )

            for chunk_index, row_start in enumerate(
                range(
                    0,
                    input_shape[0],
                    ROW_CHUNK_SIZE,
                )
            ):
                row_end = min(
                    row_start + ROW_CHUNK_SIZE,
                    input_shape[0],
                )

                raw_chunk = read_csr_rows(
                    counts_group,
                    row_start,
                    row_end,
                )

                common_chunk = raw_chunk[
                    :,
                    positions,
                ].tocsr()

                common_chunk.sum_duplicates()
                common_chunk.eliminate_zeros()
                common_chunk.sort_indices()

                chunk_path = (
                    chunk_dir
                    / f"chunk_{chunk_index:06d}.h5ad"
                )
                write_expression_chunk(
                    common_chunk,
                    obs.iloc[
                        row_start:row_end
                    ].copy(),
                    output_var,
                    chunk_path,
                )
                chunk_files.append(chunk_path)

                if (
                    chunk_index == 0
                    or (chunk_index + 1) % 25 == 0
                    or row_end == input_shape[0]
                ):
                    print(
                        f"  chunk {chunk_index + 1}: "
                        f"rows {row_start:,}:{row_end:,}"
                    )

                del raw_chunk, common_chunk
                gc.collect()

        common_uns = {
            "common_gene_subset_provenance": {
                "cache_version": CACHE_VERSION,
                "core_dataset": core_dataset,
                "source_h5ad": str(
                    harmonized_h5ad
                ),
                "n_common_genes": int(
                    len(common_gene_order)
                ),
                "common_gene_order": (
                    "alphabetical"
                ),
                "raw_counts_storage": (
                    "X and layers/counts"
                ),
                "row_chunk_size": int(
                    ROW_CHUNK_SIZE
                ),
            }
        }

        finalize_chunk_files(
            chunk_files,
            output_h5ad,
            expected_shape,
            common_gene_order,
            common_uns,
        )

        write_json_atomic(
            {
                "cache_version": CACHE_VERSION,
                "operation": (
                    "common_gene_subset"
                ),
                "source_fingerprint": (
                    source_fingerprint
                ),
                "shape": list(expected_shape),
                "var_names_sha256": (
                    strings_sha256(
                        common_gene_order
                    )
                ),
            },
            marker_path,
        )

        remove_path(chunk_dir)

    report = {
        "core_dataset": core_dataset,
        "common_subset_h5ad": str(
            output_h5ad
        ),
        "n_cells": int(expected_shape[0]),
        "n_common_genes": int(
            expected_shape[1]
        ),
        "has_counts_layer": True,
        "first_10_common_genes": ";".join(
            map(str, common_gene_order[:10])
        ),
    }

    del metadata, obs, var, output_var
    gc.collect()

    return output_h5ad, report


common_files = {}
subset_rows = []

for item in CORE_DATASETS:
    key = item["core_dataset"]

    print("\n" + "=" * 100)
    print("[COMMON SUBSET]", key)

    output_path, report = (
        build_common_gene_subset(
            key,
            harmonized_files[key],
            common_genes,
        )
    )

    common_files[key] = output_path
    subset_rows.append(report)


common_subset_report = pd.DataFrame(
    subset_rows
)
write_tsv_atomic(
    common_subset_report,
    COMMON_SUBSET_TSV,
)
display(common_subset_report)



[COMMON SUBSET] DS1_GSE103224
[BUILD common subset] DS1_GSE103224 (23207, 23982) -> (23207, 15176)
  chunk 1: rows 0:1,024
  chunk 23: rows 22,528:23,207
[concat_on_disk subprocess] 23 files -> E:\glioma-cnv-scratch\main_analysis_01\stage\DS1_GSE103224_postQC_common_genes_counts.main-analysis-01-nvme-scratch-v6-ds2-ensembl-source-fix.assembled.h5ad
[concat_on_disk kwargs] {'axis': 0, 'join': 'inner', 'merge': 'same', 'uns_merge': 'same', 'index_unique': None, 'max_loaded_elems': 3000000}
[child concat_on_disk] 23 files -> E:\glioma-cnv-scratch\main_analysis_01\stage\DS1_GSE103224_postQC_common_genes_counts.main-analysis-01-nvme-scratch-v6-ds2-ensembl-source-fix.assembled.h5ad
[child kwargs] {'axis': 0, 'index_unique': None, 'join': 'inner', 'max_loaded_elems': 3000000, 'merge': 'same', 'uns_merge': 'same'}
[PUBLISH sequential copy] E:\glioma-cnv-scratch\main_analysis_01\stage\DS1_GSE103224_postQC_common_genes_counts.main-analysis-01-nvme-scratch-v6-ds2-ensembl-source-fix.assembled.h5a

,core_dataset,common_subset_h5ad,n_cells,n_common_genes,has_counts_layer,first_10_common_genes
0,DS1_GSE103224,G:\Repository\glioma-cnv-single-cell-atlas\dat...,23207,15176,True,A1BG;A1BG-AS1;A2M;A2M-AS1;A2ML1;A4GALT;AAAS;AA...
1,DS2_GSE141383,G:\Repository\glioma-cnv-single-cell-atlas\dat...,21391,15176,True,A1BG;A1BG-AS1;A2M;A2M-AS1;A2ML1;A4GALT;AAAS;AA...
2,DS3_GSE182109,G:\Repository\glioma-cnv-single-cell-atlas\dat...,218735,15176,True,A1BG;A1BG-AS1;A2M;A2M-AS1;A2ML1;A4GALT;AAAS;AA...
3,DS4_GSE278450,G:\Repository\glioma-cnv-single-cell-atlas\dat...,88094,15176,True,A1BG;A1BG-AS1;A2M;A2M-AS1;A2ML1;A4GALT;AAAS;AA...


In [12]:
# =========================
# 11) Final on-disk concatenation
# =========================

scratch_final = (
    STAGE_DIR
    / f"{FINAL_MERGED_H5AD.stem}.{CACHE_VERSION}.assembled.h5ad"
)

remove_path(scratch_final)

print("[FINAL OUTPUT]", FINAL_MERGED_H5AD)
print("[FINAL SCRATCH ASSEMBLY]", scratch_final)

final_uns = {
    "core_4datasets_merge_info": {
        "description": (
            "Full-gene post-QC merge of four selected "
            "GBM scRNA-seq datasets; gene symbols "
            "harmonized; duplicate gene symbols summed "
            "from raw counts; common genes only."
        ),
        "datasets": dataset_keys,
        "join": "inner_common_genes",
        "raw_counts": (
            "stored in X and layers/counts"
        ),
        "gene_harmonization": (
            "DS1/DS2 Ensembl mapped with GENCODE v49; "
            "DS3/DS4 gene symbols from var_names"
        ),
        "n_cells": int(
            EXPECTED_FINAL_SHAPE[0]
        ),
        "n_genes": int(
            len(common_genes)
        ),
        "construction": (
            "direct CSR row chunks; NVMe scratch; "
            "anndata.experimental.concat_on_disk; "
            "child-process /uns patch; "
            "single sequential publication to repository"
        ),
        "cache_version": CACHE_VERSION,
    }
}

try:
    concat_on_disk_compatible(
        [
            common_files[key]
            for key in dataset_keys
        ],
        scratch_final,
    )

    patch_uns_in_child_process(
        scratch_final,
        final_uns,
    )

    final_metadata = validate_h5ad_structure(
        scratch_final,
        expected_shape=EXPECTED_FINAL_SHAPE,
        expected_var_names=common_genes,
    )
    validate_uns_mapping(
        scratch_final,
        final_uns,
    )

    if (
        final_metadata["x_nnz"]
        != final_metadata["counts_nnz"]
    ):
        raise AssertionError(
            "Final X and counts have different "
            "numbers of nonzero entries."
        )

    publish_file_cross_drive(
        scratch_final,
        FINAL_MERGED_H5AD,
    )

    final_metadata = validate_h5ad_structure(
        FINAL_MERGED_H5AD,
        expected_shape=EXPECTED_FINAL_SHAPE,
        expected_var_names=common_genes,
    )
    validate_uns_mapping(
        FINAL_MERGED_H5AD,
        final_uns,
    )

finally:
    remove_path(scratch_final)

concat_method = (
    "anndata.experimental.concat_on_disk "
    "with NVMe scratch assembly and "
    "child-process /uns patch"
)

print(
    "[MERGED]",
    FINAL_MERGED_H5AD,
    "method:",
    concat_method,
)


[FINAL OUTPUT] G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_fullgene_postQC_merged_counts.h5ad
[FINAL SCRATCH ASSEMBLY] E:\glioma-cnv-scratch\main_analysis_01\stage\GBM_core_4datasets_fullgene_postQC_merged_counts.main-analysis-01-nvme-scratch-v6-ds2-ensembl-source-fix.assembled.h5ad
[concat_on_disk subprocess] 4 files -> E:\glioma-cnv-scratch\main_analysis_01\stage\GBM_core_4datasets_fullgene_postQC_merged_counts.main-analysis-01-nvme-scratch-v6-ds2-ensembl-source-fix.assembled.h5ad
[concat_on_disk kwargs] {'axis': 0, 'join': 'inner', 'merge': 'same', 'uns_merge': 'same', 'index_unique': None, 'max_loaded_elems': 3000000}
[child concat_on_disk] 4 files -> E:\glioma-cnv-scratch\main_analysis_01\stage\GBM_core_4datasets_fullgene_postQC_merged_counts.main-analysis-01-nvme-scratch-v6-ds2-ensembl-source-fix.assembled.h5ad
[child kwargs] {'axis': 0, 'index_unique': None, 'join': 'inner', 'max_loaded_elems': 3000000, 'merge': 'same', 'uns_merge': 's

In [13]:
# =========================
# 12) Final metadata validation and reports
# =========================

final_metadata = validate_h5ad_structure(
    FINAL_MERGED_H5AD,
    expected_shape=EXPECTED_FINAL_SHAPE,
    expected_var_names=common_genes,
)

obs = final_metadata["obs"]
var = final_metadata["var"]

required_obs_columns = {
    "core_dataset",
    "dataset_id",
    "geo_id",
    "sample_id",
    "patient_id",
    "condition",
}
missing_obs_columns = (
    required_obs_columns
    - set(obs.columns)
)

if missing_obs_columns:
    raise AssertionError(
        "Final object is missing obs columns: "
        f"{sorted(missing_obs_columns)}"
    )

observed_dataset_counts = (
    obs["core_dataset"]
    .astype(str)
    .value_counts()
)

if set(observed_dataset_counts.index) != set(
    dataset_keys
):
    raise AssertionError(
        "Final object does not contain exactly the "
        "four configured core datasets."
    )

merge_summary = pd.DataFrame(
    [
        {
            "final_h5ad": str(
                FINAL_MERGED_H5AD
            ),
            "concat_method": concat_method,
            "n_cells": int(
                final_metadata["shape"][0]
            ),
            "n_genes": int(
                final_metadata["shape"][1]
            ),
            "has_counts_layer": True,
            "x_encoding": (
                final_metadata["x_encoding"]
            ),
            "counts_encoding": (
                final_metadata[
                    "counts_encoding"
                ]
            ),
            "x_nonzero_entries": (
                final_metadata["x_nnz"]
            ),
            "counts_nonzero_entries": (
                final_metadata["counts_nnz"]
            ),
            "n_core_datasets": int(
                obs["core_dataset"].nunique()
            ),
            "n_samples": int(
                obs["sample_id"].nunique()
            ),
            "n_patients": int(
                obs["patient_id"].nunique()
            ),
            "first_10_genes": ";".join(
                map(str, var.index[:10])
            ),
            "last_10_genes": ";".join(
                map(str, var.index[-10:])
            ),
            "note": (
                "Clean full-gene/common-gene post-QC "
                "merged object. Create the HVG/scVI "
                "branch from this object."
            ),
        }
    ]
)
write_tsv_atomic(
    merge_summary,
    MERGE_SUMMARY_TSV,
)


cell_counts_by_dataset = (
    obs.groupby(
        [
            "core_dataset",
            "dataset_id",
            "geo_id",
        ],
        observed=True,
    )
    .size()
    .reset_index(name="n_cells")
    .sort_values(
        "n_cells",
        ascending=False,
    )
)
write_tsv_atomic(
    cell_counts_by_dataset,
    CELL_COUNTS_BY_DATASET_TSV,
)


cell_counts_by_sample = (
    obs.groupby(
        [
            "core_dataset",
            "sample_id",
        ],
        observed=True,
    )
    .size()
    .reset_index(name="n_cells")
    .sort_values(
        [
            "core_dataset",
            "n_cells",
        ],
        ascending=[True, False],
    )
)
write_tsv_atomic(
    cell_counts_by_sample,
    CELL_COUNTS_BY_SAMPLE_TSV,
)


cell_counts_by_patient = (
    obs.groupby(
        [
            "core_dataset",
            "patient_id",
        ],
        observed=True,
    )
    .size()
    .reset_index(name="n_cells")
    .sort_values(
        [
            "core_dataset",
            "n_cells",
        ],
        ascending=[True, False],
    )
)
write_tsv_atomic(
    cell_counts_by_patient,
    CELL_COUNTS_BY_PATIENT_TSV,
)


display(merge_summary)
display(cell_counts_by_dataset)

print()
print("Saved and validated:", FINAL_MERGED_H5AD)
print(
    "Validated shape:",
    final_metadata["shape"],
)
print(
    "Validated X encoding:",
    final_metadata["x_encoding"],
)
print(
    "Validated counts encoding:",
    final_metadata["counts_encoding"],
)
print(
    "Validated core datasets:",
    sorted(
        obs["core_dataset"]
        .astype(str)
        .unique()
    ),
)
print(
    "Validated common genes:",
    len(common_genes),
)

del obs, var, final_metadata
gc.collect()


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\merged_fullgene_QC_summary_4datasets_corrected.tsv shape= (1, 15)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\merged_fullgene_cell_counts_by_dataset_4datasets_corrected.tsv shape= (4, 4)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\merged_fullgene_cell_counts_by_sample_4datasets_corrected.tsv shape= (83, 3)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\merged_fullgene_cell_counts_by_patient_4datasets_corrected.tsv shape= (50, 3)


,final_h5ad,concat_method,n_cells,n_genes,has_counts_layer,x_encoding,counts_encoding,x_nonzero_entries,counts_nonzero_entries,n_core_datasets,n_samples,n_patients,first_10_genes,last_10_genes,note
0,G:\Repository\glioma-cnv-single-cell-atlas\dat...,anndata.experimental.concat_on_disk with NVMe ...,351427,15176,True,csr_matrix,csr_matrix,739913124,739913124,4,83,50,A1BG;A1BG-AS1;A2M;A2M-AS1;A2ML1;A4GALT;AAAS;AA...,ZSWIM8;ZW10;ZWILCH;ZWINT;ZXDA;ZXDB;ZXDC;ZYG11B...,Clean full-gene/common-gene post-QC merged obj...


,core_dataset,dataset_id,geo_id,n_cells
2,DS3_GSE182109,DS3,GSE182109,218735
3,DS4_GSE278450,DS4,GSE278450,88094
0,DS1_GSE103224,DS1,GSE103224,23207
1,DS2_GSE141383,DS2,GSE141383,21391



Saved and validated: G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_fullgene_postQC_merged_counts.h5ad
Validated shape: (351427, 15176)
Validated X encoding: csr_matrix
Validated counts encoding: csr_matrix
Validated core datasets: ['DS1_GSE103224', 'DS2_GSE141383', 'DS3_GSE182109', 'DS4_GSE278450']
Validated common genes: 15176


0

## Expected successful completion

The final cell must report:

```text
Validated shape: (351427, 15176)
Validated X encoding: csr_matrix
Validated counts encoding: csr_matrix
Validated common genes: 15176
```

Primary output:

`data/processed/integrated/GBM_core_4datasets_fullgene_postQC_merged_counts.h5ad`

Generated reports:

1. `metadata/gene_harmonization_report_4datasets_corrected.tsv`
2. `metadata/gene_symbol_pairwise_intersections_4datasets_corrected.tsv`
3. `metadata/common_genes_after_harmonization_4datasets_corrected.tsv`
4. `metadata/common_gene_subset_report_4datasets_corrected.tsv`
5. `metadata/merged_fullgene_QC_summary_4datasets_corrected.tsv`
6. `metadata/merged_fullgene_cell_counts_by_dataset_4datasets_corrected.tsv`

The notebook deliberately contains no in-memory merge or final full-object
reopen step.

Transient chunk and assembly files are created under `E:\\glioma-cnv-scratch\\main_analysis_01` by default and are removed after successful publication.

If `concat_on_disk` omits `/uns`, this notebook restores it through a short-lived child process before validation and publication.
